In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import Libraries

In [ ]:
pip install transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoModel, AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback, AutoModelForSeq2SeqLM
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer, util

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# import wandb
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

# wandb.login(key=wandb_api_key)

# wandb.init(
#     project="24f3004524-t22026",
#     name="deberta-baseline-run1"
# )

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# EDA

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
train.head()

In [ ]:
train['answer'].value_counts()

In [ ]:
train.isnull().sum()

# Evaluation Metric

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

# Milestone 2

In [ ]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')

dataset = dataset.map(lambda x: {'combined_text': str(x['prompt']) + " " + str(x['A'])})

len(dataset[51]['combined_text'])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

tokenizer.vocab_size

In [ ]:
tokenizer.sep_token_id

In [ ]:
prompt_list = [str(text) for text in dataset['prompt']]

encoded = tokenizer(
                prompt_list, 
                padding='max_length', 
                truncation=True, 
                max_length=128, 
                return_tensors='pt'
)

encoded['input_ids'].shape

In [ ]:
model = AutoModel.from_pretrained('bert-base-uncased')

inputs = tokenizer(dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
last_hidden_state.shape

In [ ]:
cls_vector = last_hidden_state[0, 0, :]

sum_first_5 = cls_vector[:5].sum().item()
sum_first_5

In [ ]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

tokens = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
fusion_idx = tokens.index('fusion')

attention_matrix = outputs_attn.attentions[-1][0, 0, :, :] 

attn_weight = attention_matrix[0, fusion_idx].item()
attn_weight

In [ ]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


prompt_emb = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
opt_b_emb = st_model.encode(dataset[0]['B'], convert_to_tensor=True)

sim_score = util.cos_sim(prompt_emb, opt_b_emb).item()
sim_score

In [ ]:



dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
letters = ['A', 'B', 'C', 'D', 'E']

tfidf_preds = []
minilm_preds = []
minilm_map3_score = 0.0


for row in dataset:
    prompt_text = str(row['prompt'])
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    actual_ans = str(row['answer']) 
    
    # --- PIPELINE 1: TF-IDF Cosine Similarity ---
    vec = TfidfVectorizer()
    tfidf_matrix = vec.fit_transform([prompt_text] + options)
    tfidf_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    tfidf_top3 = [letters[idx] for idx in np.argsort(tfidf_sim)[-3:][::-1]]
    tfidf_preds.append(tfidf_top3)
    
    # --- PIPELINE 2: MiniLM Embeddings ---
    prompt_emb = st_model.encode(prompt_text, convert_to_tensor=True)
    opt_embs = st_model.encode(options, convert_to_tensor=True)
    
    minilm_sim = util.cos_sim(prompt_emb, opt_embs).cpu().numpy().flatten()
    
    minilm_top3 = [letters[idx] for idx in np.argsort(minilm_sim)[-3:][::-1]]
    minilm_preds.append(minilm_top3)
    
    if actual_ans in minilm_top3:
        rank = minilm_top3.index(actual_ans) + 1
        minilm_map3_score += 1.0 / rank


final_minilm_map3 = minilm_map3_score / len(dataset)

divergence_count = sum(
    1 for i in range(len(dataset))
    if str(dataset[i]['answer']) not in tfidf_preds[i] 
    and str(dataset[i]['answer']) in minilm_preds[i]
)

final_minilm_map3


In [ ]:
divergence_count

In [ ]:
from transformers import pipeline

zs_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_1 = dataset[1]
options = [str(row_1['A']), str(row_1['B']), str(row_1['C'])]

res_default = zs_pipe(str(row_1['prompt']), candidate_labels=options)
top_score = res_default['scores'][0]
top_score

In [ ]:
res_multi = zs_pipe(str(row_1['prompt']), candidate_labels=options, multi_label=True)

sum_default = sum(res_default['scores']) 
sum_multi = sum(res_multi['scores'])     

diff = abs(sum_default - sum_multi)
diff

In [ ]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

row_0 = dataset[0]

gen_prompt = f"Question: {row_0['prompt']}. Is the correct answer A: {row_0['A']} or B: {row_0['B']}? Answer with just the letter A or B."


inputs = tokenizer(gen_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)

exact_string = tokenizer.decode(outputs[0], skip_special_tokens=True)
exact_string

In [ ]:
# wandb.finish()

# Submission Cell

In [ ]:
# submission = pd.DataFrame({
#     "ID": test["id"],
#     "Prediction": test_predictions
# })

# submission.to_csv("submission.csv", index=False)

# submission.head()